<a href="https://colab.research.google.com/github/leo5358/PL_1142/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_Part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

安裝必要的套件

In [68]:
!pip install -q google-generativeai

In [69]:
import gspread # Added for self-containment
from google.colab import auth # Added for self-containment
from google.auth import default # Added for self-containment
from datetime import datetime # Added for self-containment

In [70]:
import gradio as gr
import pandas as pd
from google.colab import auth
from google.auth import default

# -*- coding: utf-8 -*-
import gspread
from datetime import datetime
import google.generativeai as genai
import os
import json

from google.colab import userdata
from google import genai

### 步驟 2: 導入函式庫與設定 API 金鑰

設定 Google Sheet 連線

In [71]:
# Global variables for Google Sheet connection (re-defined here for self-containment of this test cell)
# These should ideally be defined once in cell 9f9fcf48 and that cell executed.
SHEET_URL = "https://docs.google.com/spreadsheets/d/1ge-RkD_jfq4NNu5egU2Fk48pBttx6vpIvsPWIh96B_o/edit?usp=sharing"
WORKSHEET_NAME = "工作表2"
REQUIRED_COLUMNS = ["日期", "科目", "作業成績"] # Also from cell 9f9fcf48

_gc = None
_ws = None

def setup_gspread(sheet_url, worksheet_name):
    global _gc, _ws
    if _gc is None or _ws is None:
        print("--- 正在進行 Google Sheet 身份驗證和連線... ---")
        try:
            auth.authenticate_user()
            creds, _ = default()
            _gc = gspread.authorize(creds)
            sh = _gc.open_by_url(sheet_url)
            _ws = sh.worksheet(worksheet_name)
            print("--- Google Sheet 連線成功。---")
        except Exception as e:
            print(f"Google Sheet 連線失敗：{e}")
            _gc = None
            _ws = None

In [72]:
# 從 Colab Secrets 中獲取 API 金鑰，並使用 .strip() 去除可能存在的換行符號
api_key = userdata.get('gemini').strip()

# 使用獲取的金鑰配置 genai
client = genai.Client(api_key=api_key)

MODEL_ID = 'gemini-2.5-flash'

# (可選) 測試 AI 模型
try:
    response = client.models.generate_content(
        model = MODEL_ID, contents="Explain how AI works in a few words"
    )
    print(response.text)
except Exception as e:
    print(f"測試 AI 模型時發生錯誤：{e}")

AI learns patterns from data to make predictions or decisions.


### 定義 AI 摘要函式

In [73]:
'''
def get_ai_summary(grades):
    """
    呼叫 Gemini 模型來生成成績摘要與常見迷思。
    """
    # 準備給 AI 的提示
    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"
    for record in grades:
        date, subject, grade = record
        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"

    print("\n--- 正在呼叫 AI 模型生成摘要... ---")
    try:
        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)
        summary = response.text
        return summary
    except Exception as e:
        print(f"呼叫 AI 時發生錯誤：{e}")
        return "AI 摘要生成失敗。"
  '''

'\ndef get_ai_summary(grades):\n    """\n    呼叫 Gemini 模型來生成成績摘要與常見迷思。\n    """\n    # 準備給 AI 的提示\n    prompt_text = "以下是學生的成績列表，請幫我根據這些成績，產出一個簡單的摘要與常見迷思整理（不評分，只做總結）。\n\n"\n    for record in grades:\n        date, subject, grade = record\n        prompt_text += f"日期：{date}, 科目：{subject}, 成績：{grade}\n"\n\n    print("\n--- 正在呼叫 AI 模型生成摘要... ---")\n    try:\n        response = client.models.generate_content(model = MODEL_ID, contents = prompt_text)\n        summary = response.text\n        return summary\n    except Exception as e:\n        print(f"呼叫 AI 時發生錯誤：{e}")\n        return "AI 摘要生成失敗。"\n  '

In [74]:
from datetime import datetime

# 負責將成績寫入 Google Sheet
def save_scores_to_sheet(grade_data):
    global _gc, _ws
    if _ws is None:
        setup_gspread(SHEET_URL, WORKSHEET_NAME)
        if _ws is None:
            # 失敗時維持原狀，不重設清單
            return "錯誤：Google Sheet 未能成功連線。", grade_data, grade_data

    if not grade_data:
        return "提示：清單內沒有資料可供儲存。", grade_data, grade_data

    new_grades_for_sheet = []
    today = datetime.now().strftime('%Y-%m-%d')

    for subject, grade_str in grade_data:
        try:
            grade = float(grade_str)
            if 0 <= grade <= 100:
                new_grades_for_sheet.append([today, subject, grade])
            else:
                return f"錯誤：科目「{subject}」的分數不合理。", grade_data, grade_data
        except ValueError:
            return f"錯誤：科目「{subject}」的成績格式錯誤。", grade_data, grade_data

    try:
        _ws.append_rows(new_grades_for_sheet)
        return "成功：成績已寫入 Google Sheet，暫存清單已重設。", [], []
    except Exception as e:
        return f"失敗：寫入試算表時發生錯誤 {e}", grade_data, grade_data

# 負責產生 AI 摘要 (並可選擇是否寫回 Sheet)
def generate_ai_summary_only(grade_data):
    global _gc, _ws
    if _ws is None:
        setup_gspread(SHEET_URL, WORKSHEET_NAME)
        if _ws is None: return "Google Sheet 連線失敗。"

    # --- 1. 從試算表抓取所有歷史成績 ---
    history_records = []
    try:
        all_rows = _ws.get_all_values()
        # 跳過標題列並過濾掉可能誤入的摘要文字
        for row in all_rows[1:]:
            if len(row) >= 3 and row[1] not in ['科目', 'AI 摘要']:
                history_records.append(f"日期：{row[0]}, 科目：{row[1]}, 分數：{row[2]}")
    except Exception as e:
        print(f"讀取歷史失敗：{e}")

    # --- 2. 準備 Prompt (結合歷史與當前) ---
    current_text = ""
    if grade_data:
        for subject, grade in grade_data:
            current_text += f"科目：{subject}, 分數：{grade}\n"
    else:
        current_text = "本次無新輸入，請針對現有歷史資料進行趨勢回顧。"

    prompt_text = (
        "你是一位教育數據分析專家。請分析以下學生的成績趨勢：\n\n"
        "【過往紀錄】\n" + ("\n".join(history_records) if history_records else "尚無紀錄") + "\n\n"
        "【本次/最新資料】\n" + current_text + "\n\n"
        "請提供：1. 成績起伏趨勢分析 2. 進步與待加強科目建議 3. 學習迷思破解。"
    )

    # --- 3. 呼叫 Gemini 並寫入獨立工作表 ---
    try:
        response = client.models.generate_content(model=MODEL_ID, contents=prompt_text)
        summary = response.text

        # 寫入獨立分頁：避免弄髒原始成績表
        try:
            sh = _gc.open_by_url(SHEET_URL)
            try:
                summary_ws = sh.worksheet("AI分析紀錄")
            except:
                summary_ws = sh.add_worksheet(title="AI分析紀錄", rows="100", cols="3")
                summary_ws.append_row(["日期", "類型", "內容摘要"])

            summary_ws.append_row([datetime.now().strftime('%Y-%m-%d'), "歷史趨勢分析", summary])
        except:
            pass

        return summary
    except Exception as e:
        return f"AI 分析失敗：{e}"

In [75]:
import pandas as pd
import plotly.express as px

# 負責從 Google Sheet 抓取資料並繪製動態圖表的函式
def update_chart():
    global _ws
    # 確保連線正常
    if _ws is None:
        setup_gspread(SHEET_URL, WORKSHEET_NAME)
        if _ws is None:
            return None

    try:
        all_rows = _ws.get_all_values()

        # 資料清洗：將文字轉換為可以畫圖的結構化資料
        data = []
        for row in all_rows:
            # 確保欄位足夠，且跳過標題行或 AI 摘要
            if len(row) >= 3 and row[1] not in ['科目', 'AI 摘要']:
                try:
                    date_str = row[0]
                    subject = row[1]
                    score = float(row[2])
                    data.append([date_str, subject, score])
                except ValueError:
                    continue # 忽略無法轉換為數字的異常資料

        if not data:
            return None # 如果沒有歷史成績，就不回傳圖表

        # 使用 pandas 將資料轉為 DataFrame
        df = pd.DataFrame(data, columns=['日期', '科目', '成績'])

        # 將日期轉為真正的時間格式，並依照時間排序，確保折線圖不會亂連
        df['日期'] = pd.to_datetime(df['日期'])
        df = df.sort_values('日期')

        # 使用 Plotly 繪製動態折線圖
        fig = px.line(
            df,
            x='日期',
            y='成績',
            color='科目',
            markers=True, # 在轉折點加上點點
            title='各科歷史成績趨勢圖',
            range_y=[0, 105], # 固定 Y 軸為 0 到 105 之間，較好閱讀
            template='plotly_white'
        )

        # 美化 X 軸日期顯示格式
        fig.update_xaxes(tickformat="%Y-%m-%d")

        return fig
    except Exception as e:
        print(f"繪製圖表失敗：{e}")
        return None

定義 Gradio 處理函式

In [76]:
import gradio as gr

SUBJECTS = ["國文", "英文", "數學", "自然", "社會"]

with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
    gr.Markdown("# 智慧成績一本通")

    # 使用 Tabs 分頁
    with gr.Tabs():

        # ========== 分頁 1：成績輸入與分析 ==========
        with gr.Tab("成績輸入與分析"):
            grade_state = gr.State([])

            with gr.Row():
                with gr.Column():
                    gr.Markdown("### 1. 輸入成績")
                    with gr.Row():
                        sub_input = gr.Dropdown(choices=SUBJECTS, label="選擇科目", value="國文")
                        num_input = gr.Number(label="成績 (0-100)", minimum=0, maximum=100, value=0)

                    add_btn = gr.Button("➕ 加入預備清單")

                    gr.Markdown("### 2. 確認預備清單")
                    display_df = gr.Dataframe(headers=["科目", "成績"], type="array", interactive=False)
                    clear_btn = gr.Button("清空清單", size="sm")

                with gr.Column():
                    gr.Markdown("### 3. 執行動作")
                    with gr.Row():
                        save_btn = gr.Button("儲存至試算表", variant="primary")
                        ai_btn = gr.Button("產生 AI 摘要", variant="secondary")

                    gr.Markdown("### 執行結果")
                    status_output = gr.Textbox(label="系統訊息")
                    summary_output = gr.Textbox(label="Gemini 分析報告", lines=12)

            # --- 處理成績輸入的邏輯 ---
            def add_logic(current_list, sub, score):
                if score is None or score < 0 or score > 100:
                    raise gr.Error("請輸入 0 到 100 之間的有效分數")
                updated = current_list + [[sub, score]]
                return updated, updated

            add_btn.click(add_logic, [grade_state, sub_input, num_input], [grade_state, display_df])
            clear_btn.click(lambda: ([], []), None, [grade_state, display_df])
            save_btn.click(fn=save_scores_to_sheet, inputs=grade_state, outputs=[status_output, grade_state, display_df])
            ai_btn.click(fn=generate_ai_summary_only, inputs=grade_state, outputs=summary_output)

        # ========== 分頁 2：視覺化圖表 ==========
        with gr.Tab("歷史趨勢圖表"):
            gr.Markdown("### 檢視各科目的成績變化趨勢")
            gr.Markdown("如果您剛剛有新增成績，請點擊下方按鈕來獲取最新圖表。")

            refresh_chart_btn = gr.Button("載入 / 更新圖表", variant="primary")

            # 用來顯示 Plotly 圖表的元件
            chart_output = gr.Plot(label="成績趨勢")

            # 按鈕觸發畫圖函式 (呼叫上一個儲存格的 update_chart)
            refresh_chart_btn.click(
                fn=update_chart,
                inputs=None,
                outputs=chart_output
            )

# 啟動介面
demo.launch()

/tmp/ipykernel_7061/3491503280.py:5: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.



It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e723f29be61964e5f4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
